# Experiment 2 — ST-GCN Joint Motion, NTU60 XSub

This notebook runs the controlled Joint Motion experiment for 16 outer epochs with `RepeatDataset(times=5)`, independently evaluates its best checkpoint, and compares its full validation predictions against the frozen Joint baseline. It automatically resumes the latest compatible full Joint Motion checkpoint found in Kaggle Input, including optimizer/scheduler state and its exact epoch history. It does not modify or retrain the Joint model.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/mzuyyy/Human-action-recognition.git'
PROJECT_DIR = Path('/kaggle/working/ntu-action-recognition')
MMACTION2_DIR = Path('/kaggle/working/mmaction2')
CONFIG_PATH = PROJECT_DIR / 'configs/stgcn_ntu60_xsub_joint_motion_80e.py'
WORK_DIR = PROJECT_DIR / 'work_dirs/stgcn_ntu60_xsub_joint_motion_80e'
ANN_FILE = PROJECT_DIR / 'data/skeleton/ntu60_2d.pkl'
BASELINE_EVAL_DIR = PROJECT_DIR / 'artifacts/evaluation'
BASELINE_CHECKPOINT = PROJECT_DIR / 'artifacts/checkpoints/stgcn_joint_ntu60_xsub_best.pth'
MOTION_DIR = PROJECT_DIR / 'artifacts/experiments/joint_motion'
COMPARISON_DIR = PROJECT_DIR / 'artifacts/comparison'
NTU60_URL = 'https://download.openmmlab.com/mmaction/v1.0/skeleton/data/ntu60_2d.pkl'

# Resume is mandatory for this continuation run. The notebook discovers the
# latest compatible full checkpoint here; no manual filename edit is needed.
RESUME_INPUT_DIR = Path(
    '/kaggle/input/models/duymaingoc/resume-1/pytorch/default/1')
REQUIRE_RESUME = True
AUTO_RESUME = True
RESUME_CHECKPOINT = None
RESUME_METRICS = None

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
for directory in (WORK_DIR, BASELINE_EVAL_DIR, MOTION_DIR, COMPARISON_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print('project:', PROJECT_DIR)


In [ ]:
%%bash
set -euo pipefail
python -m pip uninstall -q -y mmcv mmcv-lite >/dev/null 2>&1 || true
python -m pip install -q --only-binary=mmcv-lite \
  "importlib-metadata" "mmengine>=0.7.1,<1.0.0" "mmcv-lite==2.1.0"

MMACTION2_SRC=/kaggle/working/mmaction2
if [ ! -d "${MMACTION2_SRC}/.git" ]; then
  git -c advice.detachedHead=false clone --branch v1.2.0 --depth 1 \
    https://github.com/open-mmlab/mmaction2.git "${MMACTION2_SRC}"
else
  if ! git -C "${MMACTION2_SRC}" rev-parse -q --verify \
      "refs/tags/v1.2.0^{commit}" >/dev/null; then
    git -C "${MMACTION2_SRC}" fetch -q --depth 1 origin tag v1.2.0
  fi
  git -c advice.detachedHead=false -C "${MMACTION2_SRC}" \
    checkout -q --detach v1.2.0
fi
python -m pip uninstall -q -y mmaction2 >/dev/null 2>&1 || true
python -m pip install -q -e "${MMACTION2_SRC}"

python - <<'PY'
from pathlib import Path
path = Path('/kaggle/working/mmaction2/mmaction/utils/dependency.py')
text = path.read_text()
old = "WITH_MULTIMODAL = all(\n    satisfy_requirement(item) for item in ['transformers>=4.28.0'])"
new = "# Disabled for this skeleton-only environment.\nWITH_MULTIMODAL = False"
if old in text:
    path.write_text(text.replace(old, new))
elif new not in text:
    raise RuntimeError(f'Could not disable MMAction2 multimodal imports in {path}')
PY

PYTHONPATH=/kaggle/working/mmaction2:/kaggle/working/ntu-action-recognition \
python - <<'PY'
import mmaction
import mmaction.datasets
import mmaction.models
print('mmaction ->', mmaction.__version__, mmaction.__file__)
PY


In [ ]:
# Dataset is restored independently; the Joint baseline remains read-only.
import glob
import json
import shutil
import sys
import urllib.request

if ANN_FILE.is_symlink() and not ANN_FILE.exists():
    ANN_FILE.unlink()
if not ANN_FILE.exists():
    hits = glob.glob('/kaggle/input/**/ntu60_2d.pkl', recursive=True)
    ANN_FILE.parent.mkdir(parents=True, exist_ok=True)
    if hits:
        ANN_FILE.symlink_to(Path(hits[0]).resolve())
    else:
        temporary = ANN_FILE.with_suffix('.pkl.part')
        temporary.unlink(missing_ok=True)
        urllib.request.urlretrieve(NTU60_URL, temporary)
        temporary.replace(ANN_FILE)
print('dataset:', ANN_FILE)

baseline_names = (
    'baseline_metrics.json', 'predictions.csv', 'y_true.npy',
    'y_pred.npy', 'y_score.npy')
if not all((BASELINE_EVAL_DIR / name).is_file() for name in baseline_names):
    candidate_dirs = []
    for metrics_path in Path('/kaggle/input').rglob('baseline_metrics.json'):
        if all((metrics_path.parent / name).is_file() for name in baseline_names):
            candidate_dirs.append(metrics_path.parent)
    if len(candidate_dirs) == 1:
        for name in baseline_names:
            destination = BASELINE_EVAL_DIR / name
            if not destination.is_file():
                destination.symlink_to((candidate_dirs[0] / name).resolve())
    elif candidate_dirs:
        raise RuntimeError(
            'Attach exactly one complete Joint evaluation bundle; found '
            f'{candidate_dirs}')
    else:
        # A full prediction bundle was not uploaded. Recreate it from the
        # trusted final Joint checkpoint + its log; this performs inference
        # only and never retrains or modifies the frozen Joint experiment.
        preferred_joint_dir = Path(
            '/kaggle/input/models/duymaingoc/resume/pytorch/default/2')
        if ((preferred_joint_dir / 'epoch_16.pth').is_file() and
                (preferred_joint_dir / 'epoch_metrics.jsonl').is_file()):
            source_dirs = [preferred_joint_dir]
        else:
            source_dirs = sorted({
                checkpoint.parent for checkpoint in
                Path('/kaggle/input').rglob('epoch_16.pth')
                if (checkpoint.parent / 'epoch_metrics.jsonl').is_file()
            })
        if len(source_dirs) != 1:
            raise FileNotFoundError(
                'No complete Joint prediction bundle was attached, so '
                'exactly one directory containing epoch_16.pth and '
                'epoch_metrics.jsonl is required. Found: '
                f'{source_dirs}')
        baseline_environment = os.environ.copy()
        baseline_environment['PYTHONPATH'] = (
            str(MMACTION2_DIR) + os.pathsep + str(PROJECT_DIR) +
            os.pathsep + baseline_environment.get('PYTHONPATH', ''))
        baseline_environment['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
        baseline_command = [
            sys.executable, 'scripts/evaluate_stgcn_joint.py',
            '--config', 'configs/stgcn_ntu60_xsub_80e_resume.py',
            '--ann-file', str(ANN_FILE),
            '--work-dir', str(source_dirs[0]),
            '--frozen-checkpoint', str(BASELINE_CHECKPOINT),
            '--evaluation-dir', str(BASELINE_EVAL_DIR),
        ]
        print('rebuilding frozen Joint predictions from:', source_dirs[0])
        subprocess.run(
            baseline_command, cwd=PROJECT_DIR,
            env=baseline_environment, check=True)

missing_baseline = [
    name for name in baseline_names
    if not (BASELINE_EVAL_DIR / name).is_file()]
if missing_baseline:
    raise RuntimeError(
        f'Joint baseline inference bundle is incomplete: {missing_baseline}')

if not BASELINE_CHECKPOINT.is_file():
    candidates = sorted(
        Path('/kaggle/input').rglob('stgcn_joint_ntu60_xsub_best.pth'))
    if not candidates:
        preferred_checkpoint = Path(
            '/kaggle/input/models/duymaingoc/resume/pytorch/default/2/'
            'epoch_16.pth')
        if (preferred_checkpoint.is_file() and
                (preferred_checkpoint.parent /
                 'epoch_metrics.jsonl').is_file()):
            candidates = [preferred_checkpoint]
        else:
            candidates = sorted(
                checkpoint for checkpoint in
                Path('/kaggle/input').rglob('epoch_16.pth')
                if (checkpoint.parent /
                    'epoch_metrics.jsonl').is_file())
    if len(candidates) != 1:
        raise FileNotFoundError(
            'Frozen Joint checkpoint could not be identified uniquely. '
            f'Candidates: {candidates}')
    BASELINE_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(candidates[0], BASELINE_CHECKPOINT)

baseline_record = PROJECT_DIR / 'artifacts/experiments/joint/baseline.json'
baseline = json.loads(baseline_record.read_text())
assert baseline['top1'] == 0.8823 and baseline['top5'] == 0.9877
independent_baseline = json.loads(
    (BASELINE_EVAL_DIR / 'baseline_metrics.json').read_text())
if independent_baseline.get('evaluation_status') != 'accepted':
    raise RuntimeError('the independent Joint evaluation is not accepted')
if (abs(float(independent_baseline['top1']) - baseline['top1']) > .002 or
        abs(float(independent_baseline['top5']) - baseline['top5']) > .002):
    raise RuntimeError(
        'independent Joint metrics do not reproduce the frozen baseline')
if int(independent_baseline['validation_samples']) != 16487:
    raise RuntimeError('Joint evaluation did not cover all 16487 xsub_val samples')
print('frozen Joint checkpoint:', BASELINE_CHECKPOINT)
print('frozen Joint predictions:', BASELINE_EVAL_DIR)


In [ ]:
# Validate the resolved controlled config and choose fresh/resume mode.
import gc
import importlib
import re
import shutil
import sys
import torch

# The running Jupyter kernel does not re-process an editable install's new
# .pth file. Expose both source trees explicitly before custom_imports runs.
for source_dir in (PROJECT_DIR, MMACTION2_DIR):
    if str(source_dir) not in sys.path:
        sys.path.insert(0, str(source_dir))
importlib.invalidate_caches()
import mmaction
from mmengine.config import Config

mmaction_module = Path(mmaction.__file__).resolve()
if not mmaction_module.is_relative_to(MMACTION2_DIR.resolve()):
    raise RuntimeError(
        f'kernel imported stale MMAction2 module: {mmaction_module}')
cfg = Config.fromfile(str(CONFIG_PATH))
assert cfg.work_dir == 'work_dirs/stgcn_ntu60_xsub_joint_motion_80e'
assert cfg.train_cfg.max_epochs == 16
assert cfg.train_dataloader.batch_size == 64
assert cfg.train_dataloader.dataset.times == 5
assert cfg.param_scheduler[0].T_max == 16
assert cfg.randomness.seed == 42
assert cfg.load_from is None and cfg.resume is True
assert cfg.train_dataloader.dataset.dataset.pipeline[1].feats == ['jm']

def read_resume_history(path, checkpoint_epoch):
    if not path.is_file():
        raise RuntimeError(f'matching epoch_metrics.jsonl is missing: {path}')
    rows = [
        json.loads(line) for line in path.read_text().splitlines()
        if line.strip()]
    prefix = [
        row for row in rows
        if int(row['outer_epoch']) <= checkpoint_epoch]
    epochs = [int(row['outer_epoch']) for row in prefix]
    if epochs != list(range(1, checkpoint_epoch + 1)):
        raise RuntimeError(
            'metrics must contain each true outer epoch through the '
            f'checkpoint exactly once; got {epochs}')
    for row in prefix:
        if int(row['effective_epoch']) != int(row['outer_epoch']) * 5:
            raise RuntimeError(f'off-by-one resume metric: {row}')
    return prefix

def inspect_motion_checkpoint(path, metrics_path, trusted_motion=False):
    try:
        checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    except Exception as error:
        raise RuntimeError(
            f'{path}: torch.load failed: {type(error).__name__}: {error}') from error
    metadata = checkpoint.get('meta', {})
    checkpoint_config = ''.join(str(metadata.get('cfg', '')).split())
    is_motion = ("feats=['jm']" in checkpoint_config or
                 'feats=["jm"]' in checkpoint_config)
    is_joint = ("feats=['j']" in checkpoint_config or
                'feats=["j"]' in checkpoint_config)
    if is_joint and not is_motion:
        raise RuntimeError(f'{path}: metadata identifies a Joint checkpoint')
    if not is_motion and not trusted_motion:
        raise RuntimeError(f'{path}: Joint Motion metadata is absent')
    epoch_match = re.search(r'epoch[_-](\d+)', path.name)
    checkpoint_epoch = int(metadata.get(
        'epoch', epoch_match.group(1) if epoch_match else 0))
    if not 1 <= checkpoint_epoch <= 16:
        raise RuntimeError(f'{path}: invalid epoch {checkpoint_epoch}')
    if metadata.get('seed') is not None and int(metadata['seed']) != 42:
        raise RuntimeError(f'{path}: expected training seed 42')
    if ('max_epochs=' in checkpoint_config and
            'max_epochs=16' not in checkpoint_config):
        raise RuntimeError(f'{path}: checkpoint max_epochs is not 16')
    if 'T_max=' in checkpoint_config and 'T_max=16' not in checkpoint_config:
        raise RuntimeError(f'{path}: checkpoint cosine T_max is not 16')
    if not isinstance(checkpoint.get('state_dict'), dict):
        raise RuntimeError(f'{path}: model state_dict is missing')
    if not isinstance(checkpoint.get('optimizer'), dict):
        raise RuntimeError(f'{path}: optimizer state is missing')
    if not checkpoint.get('param_schedulers'):
        raise RuntimeError(f'{path}: scheduler state is missing')
    del checkpoint
    gc.collect()
    history = read_resume_history(metrics_path, checkpoint_epoch)
    return dict(
        path=path, metrics_path=metrics_path, epoch=checkpoint_epoch,
        history=history)

resume_selection = None
resume_candidates = []
resume_diagnostics = []
last_marker = WORK_DIR / 'last_checkpoint'
if RESUME_CHECKPOINT is not None:
    explicit_checkpoint = Path(RESUME_CHECKPOINT)
    explicit_metrics = (Path(RESUME_METRICS) if RESUME_METRICS else
                        explicit_checkpoint.parent / 'epoch_metrics.jsonl')
    if not explicit_checkpoint.is_file():
        raise FileNotFoundError(explicit_checkpoint)
    resume_selection = inspect_motion_checkpoint(
        explicit_checkpoint, explicit_metrics, trusted_motion=True)
elif last_marker.is_file():
    marked = Path(last_marker.read_text().strip())
    local_checkpoint = marked if marked.is_absolute() else WORK_DIR / marked.name
    if not local_checkpoint.is_file():
        raise FileNotFoundError(f'last_checkpoint target is missing: {local_checkpoint}')
    resume_selection = inspect_motion_checkpoint(
        local_checkpoint, WORK_DIR / 'epoch_metrics.jsonl',
        trusted_motion=True)
elif AUTO_RESUME:
    search_paths = list(WORK_DIR.glob('*.pth'))
    if not RESUME_INPUT_DIR.is_dir():
        raise FileNotFoundError(
            f'required resume input directory is missing: {RESUME_INPUT_DIR}')
    input_checkpoints = sorted({
        *RESUME_INPUT_DIR.rglob('*.pth'),
        *RESUME_INPUT_DIR.rglob('*.pt'),
        *RESUME_INPUT_DIR.rglob('*.ckpt'),
    })
    input_metrics = sorted(
        path for path in RESUME_INPUT_DIR.rglob('*')
        if path.is_file() and path.suffix == '.jsonl' and
        'epoch' in path.name.lower() and 'metric' in path.name.lower())
    print('resume checkpoint files:', [str(path) for path in input_checkpoints])
    print('resume metric files:', [str(path) for path in input_metrics])
    search_paths.extend(input_checkpoints)
    seen = set()
    for candidate in search_paths:
        resolved = candidate.resolve()
        if resolved in seen or resolved == BASELINE_CHECKPOINT.resolve():
            continue
        seen.add(resolved)
        adjacent_metrics = candidate.parent / 'epoch_metrics.jsonl'
        if adjacent_metrics.is_file():
            metrics_path = adjacent_metrics
        elif candidate.is_relative_to(WORK_DIR) and (
                WORK_DIR / 'epoch_metrics.jsonl').is_file():
            metrics_path = WORK_DIR / 'epoch_metrics.jsonl'
        elif len(input_metrics) == 1:
            metrics_path = input_metrics[0]
        else:
            resume_diagnostics.append(
                f'{candidate}: cannot identify one matching metrics JSONL; '
                f'found {input_metrics}')
            continue
        try:
            inspected = inspect_motion_checkpoint(
                candidate, metrics_path,
                trusted_motion=candidate.is_relative_to(RESUME_INPUT_DIR))
        except RuntimeError as error:
            resume_diagnostics.append(str(error))
            continue
        resume_candidates.append(inspected)
    if resume_candidates:
        latest_epoch = max(item['epoch'] for item in resume_candidates)
        latest = [
            item for item in resume_candidates
            if item['epoch'] == latest_epoch]
        source_dirs = {item['path'].parent.resolve() for item in latest}
        if len(source_dirs) != 1:
            raise RuntimeError(
                'multiple Joint Motion runs have the same latest epoch; '
                'set RESUME_CHECKPOINT explicitly: '
                f'{[str(item["path"]) for item in latest]}')
        regular_name = f'epoch_{latest_epoch}.pth'
        resume_selection = sorted(
            latest, key=lambda item: item['path'].name == regular_name,
            reverse=True)[0]
    elif resume_diagnostics:
        raise RuntimeError(
            'Joint Motion checkpoint(s) were found but cannot be resumed:\n' +
            '\n'.join(resume_diagnostics))

if resume_selection is None and REQUIRE_RESUME:
    raise RuntimeError(
        'resume is required, but no compatible full Joint Motion checkpoint '
        f'with adjacent epoch_metrics.jsonl was found in {RESUME_INPUT_DIR}')

resume_checkpoint = None
resume_metrics_path = None
checkpoint_epoch = None
if resume_selection is not None:
    checkpoint_epoch = resume_selection['epoch']
    resume_metrics_path = resume_selection['metrics_path']
    resume_history = resume_selection['history']
    local_metrics = WORK_DIR / 'epoch_metrics.jsonl'
    if local_metrics.is_file():
        local_history = [
            json.loads(line) for line in local_metrics.read_text().splitlines()
            if line.strip()]
        if local_history != resume_history:
            raise RuntimeError('local and selected resume histories differ')
    else:
        local_metrics.write_text(
            ''.join(json.dumps(row) + '\n' for row in resume_history))

    source_dir = resume_selection['path'].parent
    if source_dir.resolve() != WORK_DIR.resolve():
        staging_candidates = [resume_selection]
        staged_sources = {resume_selection['path'].resolve()}
        for source in source_dir.glob('*.pth'):
            if source.resolve() in staged_sources:
                continue
            try:
                item = inspect_motion_checkpoint(
                    source, resume_metrics_path, trusted_motion=True)
            except RuntimeError:
                continue
            staging_candidates.append(item)
            staged_sources.add(source.resolve())
        for item in staging_candidates:
            source = item['path']
            if (source.parent.resolve() != source_dir.resolve() or
                    item['epoch'] > checkpoint_epoch or
                    source.name == 'latest.pth'):
                continue
            destination = WORK_DIR / source.name
            if not destination.exists() and not destination.is_symlink():
                destination.symlink_to(source.resolve())
    canonical_checkpoint = WORK_DIR / f'epoch_{checkpoint_epoch}.pth'
    if not canonical_checkpoint.is_file():
        canonical_checkpoint.symlink_to(resume_selection['path'].resolve())
    latest_link = WORK_DIR / 'latest.pth'
    temporary_link = WORK_DIR / '.latest.pth.resume.tmp'
    temporary_link.unlink(missing_ok=True)
    temporary_link.symlink_to(canonical_checkpoint.name)
    temporary_link.replace(latest_link)
    last_marker.write_text(canonical_checkpoint.name)
    resume_checkpoint = canonical_checkpoint
elif list(WORK_DIR.glob('*.pth')) or (WORK_DIR / 'epoch_metrics.jsonl').exists():
    raise RuntimeError(
        'Partial local outputs exist but no matching resumable checkpoint '
        'was found. Set RESUME_CHECKPOINT and RESUME_METRICS explicitly.')

launch_mode = 'resume' if resume_checkpoint else 'fresh'
if launch_mode == 'fresh':
    assert not list(WORK_DIR.glob('*.pth'))
resolved_path = WORK_DIR / 'resolved_config.py'
resolved_path.write_text(cfg.pretty_text)
(WORK_DIR / 'launch.json').write_text(json.dumps({
    'mode': launch_mode,
    'resume_checkpoint': str(resume_checkpoint) if resume_checkpoint else None,
    'resume_metrics': str(resume_metrics_path) if resume_metrics_path else None,
    'resume_epoch': checkpoint_epoch,
    'auto_resume': AUTO_RESUME,
    'require_resume': REQUIRE_RESUME,
    'resume_input_dir': str(RESUME_INPUT_DIR),
    'seed': 42, 'outer_epochs': 16, 'repeat_times': 5,
}, indent=2))
print('launch mode:', launch_mode)
print('resume checkpoint:', resume_checkpoint)
print('resume outer epoch:', checkpoint_epoch)
print('resolved config:', resolved_path)


In [ ]:
# Train only Joint Motion. A same-config resume restores optimizer/scheduler.
import time

environment = os.environ.copy()
environment['PYTHONPATH'] = (
    str(MMACTION2_DIR) + os.pathsep + str(PROJECT_DIR) + os.pathsep
    + environment.get('PYTHONPATH', ''))
environment['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
environment['PYTHONUNBUFFERED'] = '1'
command = [
    sys.executable, str(MMACTION2_DIR / 'tools/train.py'),
    str(CONFIG_PATH), '--work-dir', str(WORK_DIR), '--seed', '42',
]
if resume_checkpoint:
    command.extend(['--resume', str(resume_checkpoint)])
console_path = WORK_DIR / 'training_console.log'
started = time.perf_counter()
with console_path.open('a' if resume_checkpoint else 'w') as console:
    process = subprocess.Popen(
        command, cwd=PROJECT_DIR, env=environment,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        bufsize=1)
    for line in process.stdout:
        print(line, end='')
        console.write(line)
        console.flush()
    return_code = process.wait()
elapsed = time.perf_counter() - started
if return_code != 0:
    raise RuntimeError(f'Joint Motion training failed with code {return_code}')
print(f'training command completed in {elapsed / 3600:.2f} hours')


In [ ]:
# Require the exact clean 1..16 history before independent evaluation.
from collections import Counter

history_path = WORK_DIR / 'epoch_metrics.jsonl'
history = [json.loads(line) for line in history_path.read_text().splitlines() if line.strip()]
counts = Counter(int(row['outer_epoch']) for row in history)
assert counts == Counter({epoch: 1 for epoch in range(1, 17)}), counts
for row in history:
    assert row['effective_epoch'] == row['outer_epoch'] * 5, row
final_checkpoint = WORK_DIR / 'epoch_16.pth'
assert final_checkpoint.is_file(), final_checkpoint
assert (WORK_DIR / 'latest.pth').is_file()
best_record = max(history, key=lambda row: float(row['val_acc_top1']))
best_epoch = int(best_record['outer_epoch'])
best_candidates = [WORK_DIR / f'epoch_{best_epoch}.pth']
best_candidates.extend(WORK_DIR.glob(
    f'best_acc_top1_epoch_{best_epoch}.pth'))
if not any(path.is_file() for path in best_candidates):
    raise RuntimeError(
        f'best logged checkpoint is epoch {best_epoch}, but that file was '
        'not included in the resume upload and no later epoch replaced it')
available_epochs = sorted(
    int(path.stem.split('_')[-1]) for path in WORK_DIR.glob('epoch_*.pth'))
print('available regular checkpoints:', available_epochs)
print('best logged outer epoch:', best_epoch)
print('| Outer | Effective | Loss | Top-1 | Top-5 | LR |')
print('|---:|---:|---:|---:|---:|---:|')
for row in sorted(history, key=lambda item: item['outer_epoch']):
    print(
        f"| {row['outer_epoch']} | {row['effective_epoch']} | "
        f"{row['train_loss']:.6f} | {row['val_acc_top1']:.4f} | "
        f"{row['val_acc_top5']:.4f} | {row['learning_rate']:.9f} |")


In [ ]:
# Freeze and independently infer the best Joint Motion checkpoint.
command = [
    sys.executable, 'scripts/evaluate_stgcn_joint.py',
    '--config', str(CONFIG_PATH), '--ann-file', str(ANN_FILE),
    '--work-dir', str(WORK_DIR),
    '--frozen-checkpoint',
    'artifacts/checkpoints/stgcn_joint_motion_ntu60_xsub_best.pth',
    '--evaluation-dir', str(MOTION_DIR),
    '--input-representation', 'joint_motion',
    '--metrics-name', 'metrics.json',
]
subprocess.run(command, cwd=PROJECT_DIR, env=environment, check=True)


In [ ]:
# Compare raw Joint and Joint Motion predictions and write the conclusion.
command = [
    sys.executable, 'scripts/analyze_joint_vs_motion.py',
    '--joint-dir', str(BASELINE_EVAL_DIR),
    '--motion-dir', str(MOTION_DIR),
    '--comparison-dir', str(COMPARISON_DIR),
    '--work-dir', str(WORK_DIR),
]
subprocess.run(command, cwd=PROJECT_DIR, env=environment, check=True)


In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown((COMPARISON_DIR / 'joint_vs_joint_motion.md').read_text()))
display(Image(filename=str(MOTION_DIR / 'training_curve.png')))
display(Image(filename=str(COMPARISON_DIR / 'confusion_joint_vs_motion.png')))
print('Joint Motion and comparison artifacts:')
for directory in (MOTION_DIR, COMPARISON_DIR):
    for path in sorted(directory.rglob('*')):
        if path.is_file():
            print('-', path.relative_to(PROJECT_DIR))
